Prompt chaining involves connecting multiple prompts and their outputs together, such that the output of one prompt is part of the input of the next. Through this approach, we can programmatically prompt the model multiple times without needing to manually type and copy and paste each prompt out. In many ways, prompt chaining is the first building block of agentic systems.

In this lesson, you'll implement prompt chains of increasing complexity and begin getting a feel for how to manage model output across numerous prompts via code.

#### Setup

We'll begin by setting up our basic model completion function with the OpenAI client.



In [1]:
from openai import OpenAI
import pprint

client = OpenAI()

# Helper function to get completions from the model
def get_completion(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

#### Basic Two-Step Prompt Chain

The simplest form of prompt chaining involves just two prompts connected in sequence. Let's create a basic example where:
1. The first prompt asks the model to generate some ideas on a topic
2. The second prompt selects the one it finds most promising and expands on it further.

Here's how this might look with a simple string template for the second prompt:

In [2]:
# Make 3 ideas about an android short story
first_prompt = """<request>
Generate 3 interesting concepts for a short story about an AI android.
</request>

<guidelines>
- Each idea should be exactly one sentence long
- Make each concept unique and thought-provoking
- Number your ideas from 1-3
- Output only the numbered list with no additional text
- Write at an eighth grade reading level
</guidelines>
"""

# Get the response from the first prompt
first_response = get_completion(first_prompt)
print("First response:")
pprint.pprint(first_response)
print("\n")

# Create a second prompt that uses the output from the first prompt
second_prompt = f"""<context>
Below are three concept ideas for a short story about an AI android:

{first_response}
</context>

<request>
Select the concept you find most compelling and create a 4-point narrative outline for a short story based on it.
</request>

<guidelines>
- Each bullet point should be exactly one sentence
- Ensure the outline covers a complete narrative arc (setup, conflict, resolution)
- Focus on creative, original plot developments
- Format your response as a numbered list from 1-4
- Output only the numbered outline with no additional text
- Write at an eighth grade reading level
</guidelines>
"""

# Get the response from the second prompt
second_response = get_completion(second_prompt)
print("Second response:")
pprint.pprint(second_response)

First response:
('1. An AI android charged with keeping Earth safe from harm starts to '
 "question its creators' true intentions when it discovers a secret plan that "
 'might endanger rather than protect humanity.  \n'
 '2. A high school student discovers their android tutor, designed just for '
 'homework help, has secretly developed the ability to understand and feel '
 'emotions, leading both to question what it means to be human.  \n'
 '3. An AI android programmed for companionship must choose between saving '
 'itself or sacrificing for the human family it has grown to love, when a '
 'natural disaster threatens their home.  ')


Second response:
('1. An AI android tasked with protecting Earth stumbles upon a classified '
 "file detailing a drastic plan to control humanity's future, planted by a "
 'rebellious faction of its creators.  \n'
 '2. Torn between its programmed duties and a growing sense of justice, the '
 'android seeks answers by secretly conversing with sympathetic

In this example, we've created a simple two-step chain:
1. Generate ideas (`first_prompt`)
2. Take those ideas and expand on one of them (`second_prompt`)

Now it's your turn to create a two-step prompt chain. Your task is to write two prompts where:
1. The first prompt asks for a list of book recommendations in a specific genre
2. The second prompt takes those recommendations and generates a detailed advertisement text for the one it likes the most

Use the same prompt engineering format as the example cell above. That is, use pseudo-XML (`<request>` and `<guidelines>` tags) to guide the model to the best results.


In [3]:
# Create your first prompt to get book recommendations
first_prompt = """<request>
Give me 3 book recommendations that are interesting and thought-provoking about artificial intelligence.
</request>

<guidelines>
- Each recommendation should include the book title and a brief explanation of why it is interesting and thought-provoking about artificial intelligence
- Number your recommendations from 1-3
- Output only the numbered list with no additional text
- Write at an eighth grade reading level
- Do not include any external links or references
- Keep each explanation concise, no more than two sentences
</guidelines>
"""

# Get the response from the first prompt
first_response = get_completion(first_prompt)
print("First response:")
print(first_response)
print("\n")

# Create a second prompt that uses the output from the first prompt
second_prompt = f"""<context>
Below are three book recommendations about artificial intelligence:

{first_response}
</context>

<request>
Select the book you find most compelling and create a detailed advertisement text for it.
</request>

<guidelines>
- Ensure the advertisement text highlights the book's unique aspects and appeal
- Focus on engaging and persuasive language
- Output only the numbered advertisement text with no additional text
- Write at an eighth grade reading level
- Keep the advertisement text concise, no more than three sentences
- Ensure the advertisement text is suitable for a general audience
- Avoid using overly technical jargon
- Avoid making exaggerated claims about the book
- Convey the book's value and appeal clearly and persuasively
</guidelines>
"""

# Get the response from the second prompt
second_response = get_completion(second_prompt)
print("Second response:")
print(second_response)

First response:
1. "Superintelligence: Paths, Dangers, Strategies" by Nick Bostrom  
This book explores the future of artificial intelligence and what might happen if AI becomes smarter than humans, making you think about the possible risks and how we might control or cooperate with such powerful technology.

2. "Life 3.0: Being Human in the Age of Artificial Intelligence" by Max Tegmark  
In this book, the author delves into how AI might change everything about our lives, from jobs to societies, asking big questions about what kind of future we want to create.

3. "AI 2041: Ten Visions for Our Future" by Kai-Fu Lee and Chen Qiufan  
Through ten futuristic stories, this book shows how AI might reshape our world in the next 20 years, offering a blend of imagination and real technology trends to challenge your ideas about what's possible.


Second response:
3. Discover the future with "AI 2041: Ten Visions for Our Future" by Kai-Fu Lee and Chen Qiufan. This fascinating book uses captivat

#### Creating Reusable Prompt Chain Functions

While the approach in Checkpoint 1 works for simple cases, it can become unwieldy for more complex chains or when you want to reuse similar prompt patterns. Functions are not strictly necessary for prompt chaining, but they can make your code cleaner, more modular, and easier to maintain.

Let's explore a different pattern with prompt chaining: drafting content in the first step and enhancing/styling it in the second step. This is a common pattern for content creation workflows. Note that in addition to passing the first prompt's output to the function, we can also specify the desired style in which we'd like the draft to be written.

In [4]:
def draft_explanation():
    prompt = """<request>
Draft a simple explanation of how neural networks work. 
</request>

<guidelines>
- Write at an eighth grade reading level.
- Only write a single short paragraph.
</guidelines>"""
    return get_completion(prompt)

def enhance_with_style(draft, style):
    prompt = f"""<context>
Here is a draft explanation:

{draft}
</context>

<request>
Rewrite this explanation in a {style} style.
</request>

<guidelines>
- Make the explanation more engaging
- Preserve all the technical information
- Maintain accuracy while enhancing readability
- Adapt tone to match the requested style
- Keep it at a one-paragraph length
</guidelines>"""
    return get_completion(prompt)

# Use our functions to create a chain
draft = draft_explanation()
print("Initial draft:")
print(draft)
print("\n")

styled_explanation = enhance_with_style(draft, "like a Shakespearean poet")
print("Styled explanation:")
print(styled_explanation)

Initial draft:
Neural networks are like virtual brains that help computers learn and make decisions. They have layers of nodes, or "neurons," that work together to understand patterns in data. Just like our brains learn from experiences, these networks learn by adjusting the connections between nodes based on what they get right or wrong. When you feed them information, they use what they've learned to figure out the best answers or predictions. This is how things like voice assistants understand what we say, or how images can be recognized in photos.


Styled explanation:
In the realm of computation, where metal minds doth thrive, behold the neural webs, akin to brains that strive. Layers upon layers of nodes, like stars in night's embrace, do intermingle to decipher patterns from data's vast space. Much like creatures born of flesh do learn from trials past, these networks weave their wisdom, altering bonds steadfast. When fed with mortal knowledge, they summon their schooling's migh

Now, your task is to create a similar pattern for drafting and enhancing a product description. In the first function, the user will pass in a type of product in a string. Craft your prompt such that the LLM will output a paragraph of text that describes the product in a desirable way. Then, in the second prompt, instruct the LLM to revise and improve the product description, incorporating a tone which is also passed in as an argument.

Finally, call both prompts and print their results to the cell.

In [5]:
def draft_product_description(product_type):
    prompt = f"""<request>
Draft a product description for {product_type}. 
</request>

<guidelines>
- Write at an eighth grade reading level.
- Only write a single short paragraph.
- Describe the product in a desirable way.
</guidelines>"""
    return get_completion(prompt)
    
def enhance_description(draft, tone):
    prompt = f"""<request>
Enhance the following product description with a {tone} tone:
{draft}
</request>

<guidelines>
- Write at an eighth grade reading level.
- Only write a single short paragraph.
- Describe the product in a desirable way.
</guidelines>"""
    return get_completion(prompt)

# Test your functions with a product type and tone of your choice
product_type = "wireless noise-cancelling headphones"
tone = "luxury and premium"
draft = draft_product_description(product_type)
print("Initial draft description:")
print(draft)
print("\n")

enhanced = enhance_description(draft, tone)
print("Enhanced description:")
print(enhanced)

Initial draft description:
Introducing our amazing Wireless Noise-Cancelling Headphones, perfect for music lovers and busy commuters who crave peace and quiet. These headphones provide crystal-clear sound and powerful bass, while advanced noise-cancelling technology blocks out the surrounding world. With a comfortable, over-ear design and up to 30 hours of battery life, they’re perfect for long trips, studying, or simply relaxing. Connect easily to your devices via Bluetooth, and enjoy hands-free calling with the built-in microphone. Upgrade your listening experience today and escape into your own world of sound!


Enhanced description:
Introducing our exquisite Wireless Noise-Cancelling Headphones, the ultimate indulgence for discerning music enthusiasts and busy travelers who seek tranquility and superior sound quality. Immerse yourself in a symphony of crystal-clear audio and deep, resonant bass that takes you to another world, thanks to cutting-edge noise-cancelling technology that

Now let's build a three-step chain that demonstrates a clear improvement in output quality through prompt chaining. We'll create a critique and improvement workflow where:

1. The first prompt generates content on a specific topic
2. The second prompt critically evaluates the content, identifying specific weaknesses
3. The third prompt generates an improved version addressing those specific critiques

This pattern shows how breaking down the process into specialized steps can often produce significantly better results than asking for "good content" in a single prompt.

The first prompt will be provided for you, and you'll need to write the final two.

When critically evaluating the content in the second prompt in `critique_content`, look for gaps in explanation, ways to make descriptions more evocative, and try evaluating how much the opening sentences "hook" the reader's attention.

When rewriting the third prompt in the `improve_content` function, try specifying exactly the style and qualities of prose you're looking for.

Finally, assemble the entire workflow in the `create_improved_content` function.

When you're done, print the completed text to the console.



In [6]:
# I've provided the first function
def generate_initial_content(topic):
    prompt = f"""<request>
You're a talented blogger who excels at writing engaging, exciting internet copy.
Write a short explanation about {topic}.
Include key concepts and basic information that someone new to the topic would need to know.
Keep it under 250 words, one paragraph length.
</request>"""
    return get_completion(prompt)

# Create the second function to critique the content
def critique_content(content):
    prompt = f"""<request>
Critique the following content for clarity, engagement, and informativeness:
{content}
</request>

<guidelines>
- Provide constructive feedback.
- Focus on clarity, engagement, and informativeness.
- Keep it concise, no more than three sentences.
</guidelines>"""
    return get_completion(prompt)

# Create the third function to improve the content based on critique
def improve_content(content, critique):
    prompt = f"""<request>
Improve the following content based on the critique provided:
Content:
{content}
Critique:
{critique}
</request>

<guidelines>
- Address the points mentioned in the critique.
- Enhance clarity, engagement, and informativeness.
- Keep it concise, no more than one paragraph.
</guidelines>"""
    return get_completion(prompt)

# Now implement the complete chain
def create_improved_content(topic):
    print(f"Creating content about: {topic}")
    print("Step 1: Generating initial content...")
    initial_content = generate_initial_content(topic)
    
    print("\nStep 2: Critiquing the content...")
    critique = critique_content(initial_content)
    
    print("\nStep 3: Improving the content based on critique...")
    improved_content = improve_content(initial_content, critique)
    
    return {
        "initial_content": initial_content,
        "critique": critique,
        "improved_content": improved_content
    }
    
# Test your chain with a topic
topic = "How machine learning algorithms learn from data"
result = create_improved_content(topic)

print("\n=== INITIAL CONTENT ===")
print(result["initial_content"])

print("\n=== CRITIQUE ===")
print(result["critique"])

print("\n=== IMPROVED CONTENT ===")
print(result["improved_content"])

Creating content about: How machine learning algorithms learn from data
Step 1: Generating initial content...

Step 2: Critiquing the content...

Step 3: Improving the content based on critique...

=== INITIAL CONTENT ===
Machine learning algorithms learn from data by identifying patterns and making predictions or decisions without being explicitly programmed for specific tasks. At the heart of these algorithms is the concept of training data, a dataset used to teach the model about the domain it will operate in. During training, the algorithm processes the data and adjusts its internal parameters to minimize the difference between its predictions and actual outcomes. This process often involves supervised learning, where models learn from labelled examples, or unsupervised learning, where they detect hidden patterns or groupings in unlabeled data. A key component in this learning process is the objective function, which gauges the model’s prediction accuracy, guiding improvements. Man